# Tách người nói + Transcribe tiếng Việt

Pipeline: `MP4 → WAV 16kHz mono → pyannote diarization → PhoWhisper/Whisper (vi) → CSV + player`

**Trước khi chạy:**
1. **Settings → Accelerator → GPU T4 x2** (hoặc P100). Chạy CPU sẽ rất chậm với model `large`.
2. **Settings → Internet → On** (cần tải model từ Hugging Face).
3. **Add-ons → Secrets →** tạo secret tên `HF_TOKEN` với token đọc từ
   <https://huggingface.co/settings/tokens>, rồi bấm *Attach* cho notebook này.
4. Chấp nhận điều khoản của model tại
   <https://huggingface.co/pyannote/speaker-diarization-community-1>.
5. Attach dataset chứa file audio, rồi sửa `INPUT_FILE` ở cell Cấu hình.

Cell 1 có thể tự **restart kernel** sau khi cài đặt — đó là hành vi bình thường, chạy tiếp
từ cell 1b. Xem mục **Sự cố thường gặp** ở cuối notebook nếu gặp lỗi import numpy.

> ⚠️ **Không hardcode token vào notebook.** Nếu token cũ từng nằm trong file này,
> hãy thu hồi nó ở trang Tokens của Hugging Face.

## 1. Cài đặt thư viện

Hai nguyên tắc để không phá vỡ môi trường:

1. **Chỉ cài package còn thiếu hoặc quá cũ** (kiểm tra cả version, không chỉ sự tồn tại —
   image Kaggle ship sẵn `pyannote.audio` 2.x không dùng được), và dùng constraints file để **ghim `numpy` / `scipy` /
   `torch` / `numba` ở đúng version đang chạy.** Nếu để pip xê dịch numpy trong lúc kernel
   đang chạy, file `.py` và extension `.so` sẽ lệch version nhau và bung lỗi kiểu
   `ImportError: cannot import name '_center' from 'numpy._core.umath'` — không sửa được
   bằng pip trong cùng session.
2. **Cài xong thì restart kernel** (cell tự làm). Vì cell này không import numpy và luôn
   restart, việc để pip đổi numpy ở đây là an toàn — nên constraints được thử theo
   **nhiều tầng, từ chặt đến lỏng**, và `torch` là thứ giữ đến cùng (bản Kaggle là build
   CUDA `+cu128`, thay bằng wheel PyPI sẽ mất GPU). Chỉ dùng một lệnh pip duy nhất để pip
   giải dependency một lần — cài nhiều lệnh liên tiếp sẽ khiến lệnh sau ghi đè version mà
   lệnh trước vừa pin.

Nếu không thiếu gì thì cell không cài và không restart. Sau khi kernel khởi động lại,
chạy tiếp từ cell **1b. Kiểm tra môi trường**.

In [ ]:
import os, re, subprocess, sys

# openai-whisper kéo theo numba/llvmlite nên dễ đụng version numpy hơn. Mặc định pipeline
# dùng PhoWhisper qua transformers, không cần package này.
INSTALL_OPENAI_WHISPER = False

# pyannote.audio PHẢI >= 4.0 vì speaker-diarization-community-1 là model của 4.x, và bản 2.x
# mà image Kaggle ship sẵn còn import `torchaudio.AudioMetaData` (đã bị xoá).
#
# LƯU Ý: pyannote.audio 4.0.x ghim cứng `torch==2.8.0`, còn image Kaggle đang có torch
# 2.10. Nên việc cài nó BẮT BUỘC hạ torch về 2.8.0 (~2-3GB tải về, vài phút). Wheel PyPI
# của torch 2.8.0 vẫn có CUDA nên GPU không mất — cell 1b sẽ xác nhận lại.
#
# Đường lùi 3.x (chỉ dùng nếu 4.x hỏng): bản 3.x còn dùng `torchaudio.AudioMetaData` đã bị
# xoá ở torchaudio >= 2.9, nên phải hạ cả torchaudio:
#     PYANNOTE_SPEC = "pyannote.audio>=3.1,<4"   PYANNOTE_MIN = "3.1"
#     EXTRA_SPECS   = ["torchaudio<2.9", "torchvision"]
#   và ở cell 2 đổi  DIAR_MODEL = "pyannote/speaker-diarization-3.1"
PYANNOTE_SPEC = "pyannote.audio>=4.0"
PYANNOTE_MIN  = "4.0"

# Cho phép tầng cuối bỏ hết constraints. Cẩn thận: pip có thể thay torch build CUDA
# (+cu128) của Kaggle bằng wheel PyPI và làm mất GPU. Chỉ bật khi mọi tầng khác đã fail.
ALLOW_UNCONSTRAINED = False

REQUIRED = [                       # (dist trên PyPI, version tối thiểu, spec cho pip)
    ("pyannote.audio", PYANNOTE_MIN, PYANNOTE_SPEC),
    ("transformers",   "4.44",       "transformers>=4.44"),
    ("soundfile",      None,         "soundfile"),
    ("pandas",         None,         "pandas"),
    # Cố tình KHÔNG cài librosa: nó kéo numba, và numba ghim trần numpy nên hay vỡ khi
    # pyannote/torch đẩy numpy lên. Pipeline chỉ cần soundfile để đọc WAV 16kHz mono.
]
if INSTALL_OPENAI_WHISPER:
    REQUIRED.append(("openai-whisper", None, "openai-whisper"))

TORCH = ["torch", "torchaudio", "torchvision"]
NUMPY = ["numpy", "scipy", "numba", "llvmlite"]

# Khi torch được phép đổi thì phải cài kèm torchaudio/torchvision (không pin) để pip kéo
# chúng về đúng bản khớp torch mới — nếu bỏ ngoài requirement set, pip giữ nguyên bản 2.10
# cũ và chúng sẽ hỏng vì đòi `torch==2.10`.
EXTRA_SPECS = ["torchaudio", "torchvision"]

# Thử lần lượt từ chặt đến lỏng: (nhãn, package ghim, spec cài thêm).
# Để pip đổi numpy/scipy là an toàn Ở ĐÂY vì cell này không import chúng và luôn restart
# kernel sau khi cài.
TIERS = [
    ("giữ nguyên torch + numpy", TORCH + NUMPY, []),
    ("giữ nguyên torch",         TORCH,         []),
    ("để pip chọn torch",        NUMPY,         EXTRA_SPECS),
    ("không ghim gì",            [],            EXTRA_SPECS),
]
if not ALLOW_UNCONSTRAINED:
    TIERS = TIERS[:-1]


def version_of(dist):
    import importlib.metadata as md
    for name in (dist, dist.replace(".", "-"), dist.replace("-", ".")):
        try:
            return md.version(name)
        except md.PackageNotFoundError:
            continue
    return None


def too_old(current, minimum):
    try:
        from packaging.version import Version
        return Version(current) < Version(minimum)
    except Exception:
        as_tuple = lambda v: [int(x) for x in re.findall(r"\d+", v)[:3]]
        return as_tuple(current) < as_tuple(minimum)


def pip_install(specs, pin_names, path):
    """Cài specs, ghim pin_names ở version đang chạy. Không dùng --quiet: nó ẩn luôn đoạn
    'The conflict is caused by' mà ta cần khi resolve thất bại."""
    pins = [f"{n}=={v}" for n in pin_names if (v := version_of(n))]
    cmd = [sys.executable, "-m", "pip", "install", *specs]

    if pins:
        with open(path, "w") as fh:
            fh.write("\n".join(pins) + "\n")
        cmd += ["-c", path]
    return subprocess.run(cmd, capture_output=True, text=True), pins


def explain(proc):
    text = (proc.stdout or "") + "\n" + (proc.stderr or "")
    found = re.search(r"The conflict is caused by:(.*?)(?:\n\nTo fix this|\Z)", text, re.S)
    return ("The conflict is caused by:" + found.group(1)).strip() if found else text[-2500:]


missing = []
for dist, min_version, spec in REQUIRED:
    current = version_of(dist)
    if current is None:
        print(f"  - {dist}: chưa có")
    elif min_version and too_old(current, min_version):
        print(f"  - {dist}: {current} quá cũ (cần >= {min_version})")
    else:
        print(f"  - {dist}: {current} OK")
        continue
    missing.append(spec)

if not missing:
    print("\nĐã có đủ thư viện — không cài gì, không cần restart.")
    print("=> Chạy tiếp cell 1b.")
else:
    work = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
    constraints = os.path.join(work, "pip-constraints.txt")
    print("\nĐang cài:", ", ".join(missing))

    proc = None
    for label, pin_names, extra in TIERS:
        if extra:
            print(f"\n[{label}] pip được phép đổi torch -> sẽ tải lại torch/torchaudio/"
                  f"torchvision (~2-3GB, vài phút). Cài kèm: {', '.join(extra)}")
        proc, pins = pip_install(missing + extra, pin_names, constraints)
        print(f"\n[{label}] ghim: {', '.join(pins) if pins else '(không có)'}"
              f" -> {'OK' if proc.returncode == 0 else f'fail (exit {proc.returncode})'}")
        if proc.returncode == 0:
            break
        print(explain(proc))

    if proc.returncode != 0:
        raise SystemExit(
            "Không tầng nào cài được. Đọc đoạn 'The conflict is caused by' ở trên rồi chọn:\n"
            "  - Nếu conflict nằm ở pyannote.audio 4.x: đổi PYANNOTE_SPEC sang "
            '"pyannote.audio>=3.1,<4", PYANNOTE_MIN="3.1", và DIAR_MODEL ở cell 2 sang '
            '"pyannote/speaker-diarization-3.1".\n'
            "  - Nếu conflict vẫn ở torch: đặt ALLOW_UNCONSTRAINED = True để bỏ hết ghim.\n"
            "  - Nếu conflict ở package khác: thêm nó vào spec với version phù hợp."
        )

    print("\nCài xong:")
    for dist, _min_version, _spec in REQUIRED:
        print(f"  - {dist}: {version_of(dist)}")
    for name in TORCH + NUMPY:
        print(f"  - {name}: {version_of(name)}")
    print("\nĐang restart kernel để nạp lại thư viện...")
    print("=> Sau khi kernel khởi động lại, chạy tiếp từ cell 1b.")
    os._exit(0)   # Kaggle tự dựng lại kernel

## 1b. Kiểm tra môi trường

Chạy cell này **sau khi kernel restart**. Nó import đúng chuỗi module đã từng bung lỗi
(`numpy.char` → `scipy.signal` → `pyannote.audio`) để phát hiện numpy lệch version ngay
tại đây, kèm hướng xử lý, thay vì để lỗi nổ ở giữa pipeline.

In [4]:
import importlib.util, traceback

HINT_NUMPY = """
numpy trong kernel này đang lệch version (file .py và extension .so không cùng một bản).
Không sửa được bằng pip trong cùng session.
  Cách xử lý: Kaggle -> menu Run -> 'Factory reset', rồi chạy lại từ cell 1.
"""

HINT_MISSING = """
Chưa cài `{name}`. Hoặc cell 1 chưa chạy, hoặc session vừa bị factory reset — reset xoá sạch
mọi package đã cài, kể cả pyannote.audio và torch 2.8.0.
  Cách xử lý: chạy cell 1 -> để nó cài và tự restart kernel -> chạy lại cell 1b.
  (Chỉ factory reset khi numpy lệch ABI; mỗi lần reset là phải tải lại ~2-3GB torch.)
"""

HINT_PYANNOTE = """
pyannote.audio đang là bản quá cũ so với torch/torchaudio của image (bản 2.x import
`torchaudio.AudioMetaData` đã bị xoá). Model speaker-diarization-community-1 cũng cần
pyannote.audio >= 4.0.
  Cách xử lý: chạy lại cell 1 — nó sẽ thấy version quá cũ và nâng cấp, rồi restart kernel.
"""


def diagnose(exc) -> str:
    text = f"{type(exc).__name__}: {exc}"
    # Thứ tự quan trọng. Dấu hiệu đặc trưng của pyannote cũ phải xét TRƯỚC nhánh
    # "chưa cài", vì `No module named 'pytorch_lightning'` cũng là ModuleNotFoundError
    # nhưng cách xử lý là nâng cấp pyannote, không phải cài pytorch_lightning.
    if any(k in text for k in ("AudioMetaData", "pytorch_lightning")):
        return HINT_PYANNOTE
    if isinstance(exc, ModuleNotFoundError):
        return HINT_MISSING.format(name=getattr(exc, "name", None) or "?")
    if any(k in text for k in ("numpy", "umath", "_core", "ABI", "binary incompat")):
        return HINT_NUMPY
    return f"Lỗi thư viện: {text}\nChạy lại cell 1 (Cài đặt)."


try:
    import numpy
    import numpy.char          # submodule lazy — chỗ lỗi `_center` thường bung ra
    import scipy.signal        # scipy compile theo ABI của numpy
    import torch
    import soundfile
    import transformers
    import pyannote.audio      # có thể bung AttributeError, không chỉ ImportError
except Exception as exc:       # noqa: BLE001 — cần bắt cả AttributeError
    traceback.print_exc()
    print("\n" + "=" * 70)
    print(diagnose(exc))
    print("=" * 70)
    raise

print("Môi trường OK.")
print(f"  numpy        {numpy.__version__}")
print(f"  scipy        {scipy.__version__}")
print(f"  torch        {torch.__version__}   CUDA: {torch.cuda.is_available()}")
try:
    import torchaudio
    print(f"  torchaudio   {torchaudio.__version__}")
except Exception as exc:
    print(f"  torchaudio   không nạp được ({exc}) — pyannote 4.x không cần nó")
print(f"  transformers {transformers.__version__}")
print(f"  pyannote     {pyannote.audio.__version__}")
# numba/librosa KHÔNG nằm trong pipeline mặc định (cell 5 đọc WAV bằng soundfile). Chỉ
# ASR_BACKEND="whisper" mới cần numba, vì openai-whisper import nó.
try:
    import numba
    print(f"  numba        {numba.__version__}")
except Exception as exc:
    print(f"  numba        lỗi ({exc}) — không sao, pipeline mặc định không dùng")
if importlib.util.find_spec("whisper") is not None:
    print("  openai-whisper: có (dùng được ASR_BACKEND='whisper')")
else:
    print("  openai-whisper: chưa cài (chỉ dùng được ASR_BACKEND='phowhisper')")
if not torch.cuda.is_available():
    print("\n  ⚠️ Không thấy GPU -> Settings -> Accelerator -> GPU T4 x2.")

Môi trường OK.
  numpy        2.5.2
  scipy        1.16.3
  torch        2.10.0+cu128   CUDA: True
  torchaudio   2.10.0+cu128
  transformers 5.0.0
  pyannote     4.0.7
  numba        lỗi (Numba needs NumPy 2.0 or less. Got NumPy 2.5.) — không sao, pipeline mặc định không dùng
  openai-whisper: chưa cài (chỉ dùng được ASR_BACKEND='phowhisper')


## 2. Cấu hình

`ASR_BACKEND`:
- `"phowhisper"` — [vinai/PhoWhisper-large](https://huggingface.co/vinai/PhoWhisper-large),
  fine-tune riêng cho tiếng Việt. **Khuyến nghị.** Cần GPU.
- `"whisper"` — `openai-whisper`, chọn size ở `WHISPER_SIZE`. Cần bật
  `INSTALL_OPENAI_WHISPER = True` ở cell 1. Dùng `large-v3` nếu có GPU; `base` chỉ để test
  cho nhanh vì chất lượng tiếng Việt rất kém (lẫn ký tự nước ngoài, lặp từ vô nghĩa).

In [5]:
import os, gc, subprocess
from collections import defaultdict
import torch

# ---- Nguồn / đích ------------------------------------------------------------
INPUT_FILE = "/kaggle/input/datasets/sonpham2805/test10/audio.mp4"     # sửa cho khớp dataset đã attach
WAV_TEMP   = "/kaggle/working/temp_diarization.wav"
OUTPUT_DIR = "/kaggle/working/speaker_chunks"
OUTPUT_CSV = "/kaggle/working/dialogues_with_audio.csv"

# ---- Tiền xử lý audio --------------------------------------------------------
# Chuỗi filter ffmpeg chạy trước mọi thứ: bỏ rumble, giảm nhiễu nền nhẹ, cân mức âm lượng
# giữa hai người nói (người ngồi xa mic bị lọt tiếng là nguồn chính của transcript rác).
# Cell 3 tự hạ xuống chuỗi đơn giản hơn nếu ffmpeg của image thiếu filter nào.
ENHANCE_AUDIO = True
AUDIO_FILTERS = [
    "highpass=f=70",                 # bỏ rumble / DC offset
    "afftdn=nr=10:nf=-30",           # giảm nhiễu nền, cố tình nhẹ để không méo tiếng
    "dynaudnorm=f=200:g=11:p=0.9",   # cân mức theo thời gian
    "alimiter=limit=0.97",           # chặn clip sau khi cân mức
]
AUDIO_FILTERS_FALLBACK = ["highpass=f=70", "dynaudnorm=f=200:g=11"]

# ---- Diarization -------------------------------------------------------------
# Phải khớp với version pyannote.audio đã cài ở cell 1:
#   pyannote.audio 4.x -> "pyannote/speaker-diarization-community-1"
#   pyannote.audio 3.x -> "pyannote/speaker-diarization-3.1"
DIAR_MODEL  = "pyannote/speaker-diarization-community-1"
MIN_SPEAKERS = 1
MAX_SPEAKERS = 5

MERGE_GAP_SEC   = 0.3    # siết từ 0.5: gộp rộng tay tạo khối dài lẫn nhiều khoảng lặng
MIN_SEGMENT_SEC = 0.8    # nâng từ 0.3: chunk dưới 1s làm Whisper bịa ra từ đơn
MAX_SEGMENT_SEC = 8.0    # hạ từ 20: khối càng dài càng dễ rơi vào vòng lặp sinh chữ
SEGMENT_PAD_SEC = 0.15   # nới hai đầu, tránh mất âm đầu/cuối do biên diarization cắt sát
REMOVE_OVERLAP  = True   # cắt bỏ phần có người khác nói chen vào giữa lượt

# ---- Tách nguồn (source separation) ------------------------------------------
# REMOVE_OVERLAP chỉ bỏ được vùng mà diarization CÓ gán nhãn người khác. Nếu nó bỏ sót
# (chỉ tìm thấy vài giây cho người thứ hai trên cả bản ghi) thì chunk vẫn còn 2 giọng.
# Tách nguồn xuất một track audio riêng cho từng người, nên chunk chỉ còn 1 giọng thật sự.
#
# Cần chấp nhận điều khoản tại https://huggingface.co/pyannote/speech-separation-ami-1.0
# Lưu ý: model train trên audio họp tiếng Anh (AMI). Trên tiếng Việt chưa chắc tốt và nó
# ngốn RAM/VRAM hơn — bật lên, nghe thử vài chunk, không đạt thì tắt lại.
USE_SEPARATION   = False
SEPARATION_MODEL = "pyannote/speech-separation-ami-1.0"

# ---- Lọc & chấm điểm --------------------------------------------------------
TRIM_SILENCE_DB  = -35.0   # ngưỡng so với đỉnh trong chunk, để cắt lặng đầu/cuối
MIN_SPEECH_SEC   = 0.5     # sau khi cắt lặng, ngắn hơn mức này thì bỏ, không cho ASR đoán
AVG_LOGPROB_MIN  = -1.0    # thấp hơn -> đánh dấu "tin cậy thấp" (ngưỡng quen dùng của Whisper)
MAX_WORD_RUN     = 2       # thu gọn chuỗi từ lặp dài hơn mức này
MIN_UNIQUE_RATIO = 0.4     # tỉ lệ từ khác nhau thấp hơn -> đánh dấu "lặp"
COMPRESSION_MAX  = 1.30    # tỉ lệ nén zlib cao hơn -> lặp xen kẽ (xem compression_ratio)

# ---- ASR ---------------------------------------------------------------------
ASR_BACKEND  = "phowhisper"          # "phowhisper" | "whisper"
PHOWHISPER   = "vinai/PhoWhisper-large"
WHISPER_SIZE = "large-v3"            # chỉ dùng khi ASR_BACKEND == "whisper"
LANGUAGE     = "vi"
REPETITION_PENALTY = 1.25   # tăng từ 1.15
NO_REPEAT_NGRAM    = 3      # hạ từ 4: "ăn ăn ăn ăn" đúng bằng 4-gram nên ngưỡng 4 không chặn

SAMPLE_RATE = 16000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)


def get_hf_token() -> str:
    """Lấy token từ Kaggle Secrets, hoặc biến môi trường HF_TOKEN khi chạy ngoài Kaggle."""
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Không tìm thấy HF_TOKEN.\n"
            "  Trên Kaggle: Add-ons -> Secrets -> New secret, Label = HF_TOKEN, "
            "value = token từ https://huggingface.co/settings/tokens, rồi Attach.\n"
            "  Ngoài Kaggle: export HF_TOKEN=hf_xxx trước khi mở notebook."
        ) from exc


HF_TOKEN = get_hf_token()
# Đẩy vào env để huggingface_hub / transformers cũng dùng, không chỉ pyannote — nếu không
# sẽ có cảnh báo "sending unauthenticated requests to the HF Hub" và bị giới hạn tốc độ.
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", HF_TOKEN)

print(f"Device: {DEVICE}")
if DEVICE == "cpu":
    print("  ⚠️ Không thấy GPU. Bật Settings -> Accelerator -> GPU, "
          "hoặc đổi WHISPER_SIZE='small' để đỡ chậm.")
print(f"ASR backend: {ASR_BACKEND}")
print(f"Tiền xử lý audio: {'bật' if ENHANCE_AUDIO else 'tắt'}")
print(f"Segment: {MIN_SEGMENT_SEC}-{MAX_SEGMENT_SEC}s, gộp khi cách < {MERGE_GAP_SEC}s, "
      f"nới biên {SEGMENT_PAD_SEC}s, cắt chồng lấn: {REMOVE_OVERLAP}")
print(f"Tách nguồn: {'bật — ' + SEPARATION_MODEL if USE_SEPARATION else 'tắt'}")
print(f"HF_TOKEN: OK ({len(HF_TOKEN)} ký tự, không in ra)")

Device: cuda
ASR backend: phowhisper
Tiền xử lý audio: bật
Segment: 0.8-8.0s, gộp khi cách < 0.3s, nới biên 0.15s, cắt chồng lấn: True
Tách nguồn: tắt
HF_TOKEN: OK (37 ký tự, không in ra)


## 3. Trích xuất WAV 16kHz mono + làm sạch audio

Chuỗi filter ffmpeg (`AUDIO_FILTERS` ở cell 2) chạy ngay tại bước này, vì mọi thứ phía sau
— diarization và ASR — đều đọc từ file WAV đã xử lý:

| filter | tác dụng |
|---|---|
| `highpass=f=70` | bỏ rumble và DC offset |
| `afftdn=nr=10:nf=-30` | giảm nhiễu nền, cố tình để nhẹ vì denoise mạnh làm méo phụ âm |
| `dynaudnorm=f=200:g=11:p=0.9` | cân mức theo thời gian — người ngồi xa mic là nguồn chính của transcript rác |
| `alimiter=limit=0.97` | chặn clip sau khi cân mức |

ffmpeg trên image có thể thiếu `afftdn` hoặc `alimiter`, nên cell thử **3 tầng**: đầy đủ →
đơn giản (`highpass` + `dynaudnorm`) → không filter. Cuối cùng in mức đỉnh/RMS dBFS để tự
kiểm: gần `0 dBFS` là đang clip, dưới `-30 dBFS` là tiếng nói quá nhỏ so với nhiễu.

In [6]:
import numpy as np
import soundfile as sf

if not os.path.exists(INPUT_FILE):
    print(f"Không tìm thấy: {INPUT_FILE}\nCác file đang có trong /kaggle/input:")
    found = False
    for root, _dirs, files in os.walk("/kaggle/input"):
        for f in files:
            print("   ", os.path.join(root, f))
            found = True
    if not found:
        print("    (trống — chưa attach dataset nào)")
    raise FileNotFoundError(INPUT_FILE)


def run_ffmpeg(filters):
    """Trích audio mono SAMPLE_RATE, áp chuỗi filter nếu có. Giữ stderr để lỗi có lý do."""
    cmd = ["ffmpeg", "-y", "-i", INPUT_FILE, "-vn", "-ar", str(SAMPLE_RATE), "-ac", "1"]
    if filters:
        cmd += ["-af", ",".join(filters)]
    cmd += ["-c:a", "pcm_s16le", WAV_TEMP]
    return subprocess.run(cmd, capture_output=True, text=True)


# ffmpeg của image có thể thiếu afftdn/alimiter -> thử 3 tầng, từ đầy đủ đến không filter.
if ENHANCE_AUDIO:
    tiers = [("đầy đủ", AUDIO_FILTERS), ("đơn giản", AUDIO_FILTERS_FALLBACK), ("thô", [])]
else:
    tiers = [("thô", [])]

print(f"Đang trích xuất audio {SAMPLE_RATE}Hz mono từ {os.path.basename(INPUT_FILE)}...")
applied = None
for label, filters in tiers:
    proc = run_ffmpeg(filters)
    if proc.returncode == 0:
        applied = (label, filters)
        break
    tail = proc.stderr.strip().splitlines()
    print(f"  [{label}] ffmpeg fail (exit {proc.returncode}): {tail[-1] if tail else '?'}")

if applied is None:
    raise RuntimeError(f"ffmpeg không chạy được cả khi không filter:\n{proc.stderr[-2000:]}")

label, filters = applied
print(f"-> filter [{label}]: {','.join(filters) if filters else '(không áp filter)'}")


# ============================================================
# Nạp audio + tiện ích xử lý (dùng lại ở cell 4 và 5)
# ============================================================
# KHÔNG dùng librosa: nó import numba, mà numba ghim trần numpy ("Numba needs NumPy 2.0
# or less"), rất dễ vỡ khi torch/pyannote đẩy numpy lên bản mới. WAV đã do ffmpeg tạo ở
# đúng SAMPLE_RATE mono nên soundfile là đủ.
audio, sr = sf.read(WAV_TEMP, dtype="float32", always_2d=False)
if audio.ndim > 1:                       # phòng trường hợp file không phải mono
    audio = audio.mean(axis=1)
if sr != SAMPLE_RATE:                    # ffmpeg đã lo, đây chỉ là lưới an toàn
    from math import gcd
    from scipy.signal import resample_poly
    g = gcd(int(sr), SAMPLE_RATE)
    audio = resample_poly(audio, SAMPLE_RATE // g, int(sr) // g)
    print(f"   resample {sr} -> {SAMPLE_RATE}Hz")
    sr = SAMPLE_RATE
audio = np.ascontiguousarray(audio, dtype=np.float32)
AUDIO_DURATION = len(audio) / sr

RMS_WIN_MS, RMS_HOP_MS = 25, 10


def frame_rms(x, sr_=None, win_ms=RMS_WIN_MS, hop_ms=RMS_HOP_MS):
    """RMS theo khung, bằng numpy thuần."""
    sr_ = sr if sr_ is None else sr_
    win = max(1, int(sr_ * win_ms / 1000))
    hop = max(1, int(sr_ * hop_ms / 1000))
    if len(x) < win:
        return np.array([float(np.sqrt(np.mean(np.square(x, dtype=np.float64)))) + 1e-12])
    frames = np.lib.stride_tricks.sliding_window_view(x, win)[::hop]
    return np.sqrt(np.square(frames, dtype=np.float64).mean(axis=1)) + 1e-12


def dbfs(x):
    if len(x) == 0:
        return -99.0
    peak = float(np.abs(x).max())
    return round(20 * float(np.log10(peak)), 1) if peak > 0 else -99.0


def quietest_point(lo, hi):
    """Thời điểm lặng nhất trong [lo, hi] — dùng để cắt segment dài mà không cắt giữa từ."""
    i0, i1 = max(0, int(lo * sr)), min(len(audio), int(hi * sr))
    if i1 - i0 < int(0.05 * sr):
        return (lo + hi) / 2.0
    rms = frame_rms(audio[i0:i1])
    hop = max(1, int(sr * RMS_HOP_MS / 1000))
    win = max(1, int(sr * RMS_WIN_MS / 1000))
    return lo + (int(np.argmin(rms)) * hop + win / 2) / sr


def extract_chunk(intervals, source=None):
    """Nối các khoảng cần giữ, fade 5ms hai đầu mỗi mảnh để chỗ nối không kêu tách.

    `source` là track riêng của speaker khi bật tách nguồn; None thì dùng audio trộn.
    """
    x = audio if source is None else source
    n_fade = max(1, int(sr * 5 / 1000))
    ramp = np.linspace(0.0, 1.0, n_fade, dtype=np.float32)
    parts = []
    for a, b in intervals:
        i0, i1 = max(0, int(a * sr)), min(len(x), int(b * sr))
        if i1 - i0 <= 2 * n_fade:
            continue
        piece = x[i0:i1].copy()
        piece[:n_fade] *= ramp
        piece[-n_fade:] *= ramp[::-1]
        parts.append(piece)
    return np.concatenate(parts) if parts else np.zeros(0, dtype=np.float32)


def trim_silence(x, rel_db=None, margin_ms=50):
    """Cắt lặng đầu/cuối theo ngưỡng tương đối so với đỉnh trong chính chunk đó."""
    rel_db = TRIM_SILENCE_DB if rel_db is None else rel_db
    if len(x) == 0:
        return x
    rms = frame_rms(x)
    above = np.nonzero(rms >= rms.max() * (10 ** (rel_db / 20)))[0]
    if len(above) == 0:
        return x[:0]
    hop = max(1, int(sr * RMS_HOP_MS / 1000))
    win = max(1, int(sr * RMS_WIN_MS / 1000))
    margin = int(sr * margin_ms / 1000)
    i0 = max(0, int(above[0]) * hop - margin)
    i1 = min(len(x), int(above[-1]) * hop + win + margin)
    return x[i0:i1]


peak_db = dbfs(audio)
rms_lin = float(np.sqrt(np.square(audio, dtype=np.float64).mean())) if len(audio) else 0.0
rms_db = round(20 * float(np.log10(rms_lin)), 1) if rms_lin > 0 else -99.0
print(f"-> {AUDIO_DURATION:.1f}s @ {sr}Hz, đỉnh {peak_db:.1f} dBFS, RMS {rms_db:.1f} dBFS")
if peak_db > -0.5:
    print("   ⚠️ Gần clip. Giảm dynaudnorm g= hoặc hạ alimiter limit=.")
if rms_db < -30:
    print("   ⚠️ RMS thấp, tiếng nói có thể quá nhỏ so với nhiễu.")

Đang trích xuất audio 16000Hz mono từ audio.mp4...
-> filter [đầy đủ]: highpass=f=70,afftdn=nr=10:nf=-30,dynaudnorm=f=200:g=11:p=0.9,alimiter=limit=0.97
-> 47.3s @ 16000Hz, đỉnh -0.6 dBFS, RMS -18.8 dBFS


## 4. Phân tách người nói (diarization)

Bốn bước, theo đúng thứ tự này:

1. **`merge_by_speaker`** — gộp theo *từng speaker riêng*, không so với segment liền trước
   trong danh sách. Danh sách xếp theo thời gian và đan xen người nói, nên chuỗi
   `A → B → A` sẽ không bao giờ gộp được hai đoạn `A` dù chúng chỉ cách nhau 0.1s.
2. **`split_long`** — cắt đoạn dài hơn `MAX_SEGMENT_SEC` (8s) **tại điểm lặng nhất trong
   audio**, không cắt đều theo mốc thời gian (cắt đều rất dễ rơi vào giữa một từ và làm
   rác cả hai nửa). `quietest_point()` tìm khung RMS nhỏ nhất trong dải 30%–70%.
3. **`pad_segments`** — nới `SEGMENT_PAD_SEC` (0.15s) hai đầu. Biên diarization hay cắt sát
   làm mất âm đầu/cuối.
4. **`subtract_overlaps`** — bỏ mọi khoảng có **speaker khác** cùng nói. Đây là xử lý gốc
   của các câu rác: ở lần chạy trước, hai segment tệ nhất đều là chunk của `SPEAKER_00`
   có lẫn giọng `SPEAKER_01` chen vào giữa. Segment bị chia thành nhiều mảnh sẽ được nối
   lại ở cell 5 với fade 5ms mỗi chỗ nối.

Ngưỡng đã siết so với bản đầu: `MERGE_GAP_SEC` 0.5 → **0.3**, `MIN_SEGMENT_SEC` 0.3 → **0.8**,
`MAX_SEGMENT_SEC` 20 → **8**. Gộp rộng tay tạo khối dài lẫn nhiều khoảng lặng, và chunk dưới
1 giây làm Whisper bịa ra từ đơn.

In [7]:
from pyannote.audio import Pipeline

def load_pipeline(name):
    pipe = Pipeline.from_pretrained(name, token=HF_TOKEN)
    if pipe is None:
        raise RuntimeError(
            f"Pipeline.from_pretrained trả về None — thường là do chưa chấp nhận điều khoản "
            f"tại https://huggingface.co/{name} hoặc token không có quyền đọc."
        )
    return pipe.to(torch.device(DEVICE))


def run_pipeline(pipe, path):
    """Gọi pipeline, bỏ min/max_speakers nếu pipeline đó không nhận."""
    try:
        return pipe(path, min_speakers=MIN_SPEAKERS, max_speakers=MAX_SPEAKERS)
    except TypeError:
        return pipe(path)


def unpack(result):
    """pyannote 4.x trả object; một số pipeline trả tuple (diarization, sources)."""
    if isinstance(result, tuple):
        annotation_, sources_ = (result + (None,))[:2]
    else:
        annotation_ = getattr(result, "speaker_diarization",
                              getattr(result, "annotation", result))
        sources_ = getattr(result, "sources", None)
    return annotation_, sources_


SPEAKER_TRACKS = {}          # speaker -> track audio riêng, rỗng nghĩa là dùng audio gốc

if USE_SEPARATION:
    print(f"Đang tải {SEPARATION_MODEL} (tách nguồn)...")
    try:
        sep_pipeline = load_pipeline(SEPARATION_MODEL)
        annotation, sources = unpack(run_pipeline(sep_pipeline, WAV_TEMP))
        if sources is None:
            raise RuntimeError("pipeline không trả về `sources`")
        data = getattr(sources, "data", sources)
        labels = list(annotation.labels())
        if data.ndim != 2 or data.shape[1] < len(labels):
            raise RuntimeError(f"hình dạng sources lạ: {getattr(data, 'shape', None)} "
                               f"với {len(labels)} speaker")
        # kênh thứ k ứng với annotation.labels()[k]
        for k, label in enumerate(labels):
            SPEAKER_TRACKS[label] = np.ascontiguousarray(data[:, k], dtype=np.float32)
        print(f"-> tách được {len(SPEAKER_TRACKS)} track: {', '.join(labels)}")
        del sep_pipeline
    except Exception as exc:
        print(f"-> tách nguồn thất bại ({type(exc).__name__}: {exc})")
        print("   Quay lại diarization thường + cắt chồng lấn theo thời gian.")
        SPEAKER_TRACKS = {}

if not SPEAKER_TRACKS:
    print(f"Đang tải {DIAR_MODEL}...")
    diar_pipeline = load_pipeline(DIAR_MODEL)
    print("Đang phân tách timeline giọng nói...")
    annotation, _sources = unpack(run_pipeline(diar_pipeline, WAV_TEMP))
    del diar_pipeline

raw_segments = [
    {"start": round(turn.start, 2), "end": round(turn.end, 2), "speaker": speaker}
    for turn, _, speaker in annotation.itertracks(yield_label=True)
]


def merge_by_speaker(segments, max_gap=MERGE_GAP_SEC, max_len=MAX_SEGMENT_SEC):
    """Gộp các lượt liền kề của cùng một người nói, cách nhau dưới max_gap giây.

    Gộp theo TỪNG speaker, không so với segment liền trước trong danh sách: danh sách xếp
    theo thời gian và đan xen người nói, nên chuỗi A -> B -> A sẽ không bao giờ gộp được
    hai đoạn A dù chúng chỉ cách nhau 0.1s.
    """
    by_speaker = defaultdict(list)
    for seg in segments:
        by_speaker[seg["speaker"]].append(dict(seg))   # copy: không sửa input

    merged = []
    for turns in by_speaker.values():
        turns.sort(key=lambda s: s["start"])
        current = None
        for seg in turns:
            if current is None:
                current = dict(seg)
                continue
            gap = seg["start"] - current["end"]
            fits = (seg["end"] - current["start"]) <= max_len
            if gap < max_gap and fits:
                current["end"] = max(current["end"], seg["end"])
            else:
                merged.append(current)
                current = dict(seg)
        if current is not None:
            merged.append(current)

    merged.sort(key=lambda s: (s["start"], s["speaker"]))
    return merged


def split_long(segments, max_len=MAX_SEGMENT_SEC):
    """Cắt segment dài hơn max_len tại ĐIỂM LẶNG NHẤT trong audio.

    Cắt đều theo mốc thời gian rất dễ rơi vào giữa một từ, và cả hai nửa đều thành rác.
    quietest_point() (định nghĩa ở cell 3) tìm khung RMS nhỏ nhất trong dải 30%..70% của
    segment, nên điểm cắt luôn nằm ở chỗ im nhất và mỗi nửa luôn >= 30% độ dài -> đệ quy
    chắc chắn dừng.
    """
    def cut(seg):
        duration = seg["end"] - seg["start"]
        if duration <= max_len:
            return [dict(seg)]
        low = seg["start"] + 0.3 * duration
        high = seg["end"] - 0.3 * duration
        point = min(max(quietest_point(low, high), low), high)
        return (cut({**seg, "end": round(point, 3)})
                + cut({**seg, "start": round(point, 3)}))

    out = []
    for seg in segments:
        out.extend(cut(seg))
    return out


def pad_segments(segments, pad, duration):
    """Nới hai đầu mỗi segment. Biên của diarization hay cắt sát, mất âm đầu/cuối."""
    return [{**seg,
             "start": round(max(0.0, seg["start"] - pad), 3),
             "end":   round(min(duration, seg["end"] + pad), 3)}
            for seg in segments]


def subtract_overlaps(segments):
    """Với mỗi segment, bỏ các khoảng thời gian có speaker KHÁC cùng nói.

    Trả về thêm khoá "keep": danh sách khoảng cần giữ. ASR nghe hai giọng chồng nhau là
    một nguồn hallucination lớn, nên thà mất một ít tiếng còn hơn cho nó đoán.

    NHƯNG không cắt đến mức xoá segment: lượt chen ngang ngắn thường nằm TRỌN trong lượt
    dài của người kia, cắt thẳng tay là mất luôn cả một speaker. Khi phần còn lại ngắn hơn
    MIN_SEGMENT_SEC thì giữ nguyên segment và chỉ đánh dấu "overlapped".
    """
    result = []
    for i, seg in enumerate(segments):
        keep = [(seg["start"], seg["end"])]
        for j, other in enumerate(segments):
            if j == i or other["speaker"] == seg["speaker"]:
                continue
            a, b = other["start"], other["end"]
            trimmed = []
            for x, y in keep:
                if b <= x or a >= y:          # không giao nhau
                    trimmed.append((x, y))
                    continue
                if a > x:                      # còn phần trước khi người kia nói
                    trimmed.append((x, min(a, y)))
                if b < y:                      # còn phần sau khi người kia dứt
                    trimmed.append((max(b, x), y))
            keep = [(x, y) for x, y in trimmed if y - x > 1e-6]

        kept_sec = sum(y - x for x, y in keep)
        full = seg["end"] - seg["start"]
        overlapped = kept_sec < full - 1e-3
        if kept_sec < MIN_SEGMENT_SEC:      # cắt xong thì không còn gì -> giữ nguyên
            keep, kept_sec = [(seg["start"], seg["end"])], full
        result.append({**seg, "keep": keep, "kept_sec": round(kept_sec, 3),
                       "overlapped": overlapped})
    return result


segments = split_long(merge_by_speaker(raw_segments))
segments = [s for s in segments if (s["end"] - s["start"]) >= MIN_SEGMENT_SEC]
after_merge = len(segments)

segments = pad_segments(segments, SEGMENT_PAD_SEC, AUDIO_DURATION)

if SPEAKER_TRACKS:
    # Mỗi speaker đã có track riêng nên chunk không còn giọng người khác. Cắt chồng lấn
    # lúc này chỉ làm mất tiếng vừa tách được, nên bỏ qua.
    print("-> đã tách nguồn, bỏ qua bước cắt chồng lấn theo thời gian")
    for s in segments:
        s["keep"] = [(s["start"], s["end"])]
        s["kept_sec"] = round(s["end"] - s["start"], 3)
        s["overlapped"] = False
elif REMOVE_OVERLAP:
    segments = subtract_overlaps(segments)
    lost = sum((s["end"] - s["start"]) - s["kept_sec"] for s in segments)
    trimmed_n = sum(1 for s in segments if s["kept_sec"] < (s["end"] - s["start"]) - 1e-3)
    kept_whole = sum(1 for s in segments if s.get("overlapped") and s["kept_sec"]
                     >= (s["end"] - s["start"]) - 1e-3)
    print(f"-> chồng lấn: cắt {trimmed_n} segment (bỏ {lost:.2f}s), "
          f"giữ nguyên {kept_whole} segment vì cắt xong sẽ không còn gì")
else:
    for s in segments:
        s["keep"] = [(s["start"], s["end"])]
        s["kept_sec"] = round(s["end"] - s["start"], 3)
        s["overlapped"] = False

speakers = sorted({s["speaker"] for s in segments})
total = sum(s["kept_sec"] for s in segments)
print(f"-> {len(raw_segments)} lượt thô -> {after_merge} sau gộp/cắt -> {len(segments)} segment "
      f"dùng được ({len(speakers)} người: {', '.join(speakers)}, tổng {total:.1f}s tiếng nói)")
frag = [s for s in segments if len(s["keep"]) > 1]
if frag:
    print(f"   {len(frag)} segment bị chồng lấn chia thành nhiều mảnh, sẽ nối lại có fade")

# Giải phóng VRAM trước khi nạp model ASR
del annotation
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Đang tải pyannote/speaker-diarization-community-1...


config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

embedding/pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

Đang phân tách timeline giọng nói...


/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  std = sequences.std(dim=-1, correction=1)


-> chồng lấn: cắt 1 segment (bỏ 3.42s), giữ nguyên 1 segment vì cắt xong sẽ không còn gì
-> 22 lượt thô -> 11 sau gộp/cắt -> 11 segment dùng được (2 người: SPEAKER_00, SPEAKER_01, tổng 38.4s tiếng nói)
   1 segment bị chồng lấn chia thành nhiều mảnh, sẽ nối lại có fade


## 5. Transcribe + chấm điểm tin cậy

Bốn lớp chống hallucination, từ trước ra sau:

1. **Cắt lặng** (`trim_silence`) theo ngưỡng tương đối `TRIM_SILENCE_DB` so với đỉnh trong
   chính chunk đó, rồi **chặn trước ASR**: chunk còn dưới `MIN_SPEECH_SEC` (0.5s) thì bỏ
   luôn, không cho model đoán. Đây là chỗ sinh ra `"loại."`, `"này."`, `"tu thuốc."`.
2. **Tham số sinh chữ**: `repetition_penalty` 1.15 → **1.25**, `no_repeat_ngram_size`
   4 → **3**. Ngưỡng 4 không chặn được `ăn ăn ăn ăn` vì chuỗi đó *đúng bằng* một 4-gram.
3. **Thu gọn lặp sau khi sinh** (`collapse_repeats`): `ăn ăn ăn ăn` → `ăn ăn`. Giữ dạng
   lặp đôi vì tiếng Việt có từ láy thật.
4. **Bắt lặp xen kẽ**: `unique_ratio` chỉ thấy lặp liên tiếp, nên chuỗi kiểu
   `tút tút loại tút thôi tút sút sút tút sút` (uniq 0.58) thoát được. Thêm tỉ lệ nén
   `zlib` — cùng họ heuristic Whisper dùng. Ngưỡng 1.30 chọn từ output thật của lần chạy
   trước: câu dùng được đo 1.02–1.15, câu rác đo 1.45 và 2.10.
5. **Chấm điểm**: `avg_logprob` mỗi segment qua `compute_transition_scores`. Dưới
   `AVG_LOGPROB_MIN` (−1.0) thì đánh dấu `tin cậy thấp` — **đánh dấu, không xoá**, để bạn
   tự nghe lại và quyết định. Tỉ lệ từ khác nhau dưới `MIN_UNIQUE_RATIO` thì đánh dấu `lặp`.

Audio truyền **trực tiếp dưới dạng numpy array**, không ghi file rồi để ASR đọc lại từ đĩa.
Các tiện ích `frame_rms` / `trim_silence` / `extract_chunk` nằm ở cell 3 (nơi audio được nạp,
dùng chung với cell 4) và viết bằng numpy thuần để không phải kéo librosa — và qua đó không
kéo numba.

In [8]:
import zlib

# ============================================================
# Tiện ích cho text  (tiện ích audio đã định nghĩa ở cell 3)
# ============================================================
PUNCT = ".,!?;:…\"')("


def word_key(word):
    """Dạng so sánh của một từ: bỏ dấu câu bám ngoài, hạ chữ thường.

    Không chuẩn hoá thì 'ăn.' và 'ăn' bị coi là hai từ khác nhau — chuỗi lặp ở cuối câu
    sẽ thoát cả bộ thu gọn lẫn bộ đếm tỉ lệ từ khác nhau.
    """
    return word.strip(PUNCT).lower()


def collapse_repeats(text, max_run=MAX_WORD_RUN):
    """Thu gọn chuỗi từ lặp liên tiếp: 'ăn ăn ăn ăn.' -> 'ăn ăn.'

    Giữ lại dạng lặp đôi vì tiếng Việt có từ láy thật ('lăm lăm', 'ào ào').
    """
    body = text.rstrip()
    tail = ""
    while body and body[-1] in PUNCT:      # tách dấu câu cuối ra rồi gắn lại sau
        tail = body[-1] + tail
        body = body[:-1]

    out = []
    for word in body.split():
        key = word_key(word)
        if key and len(out) >= max_run and all(word_key(w) == key for w in out[-max_run:]):
            # Từ bị loại có thể mang dấu câu ("này này này, rồi") -> chuyển sang từ trước,
            # nếu không dấu phẩy đó mất luôn.
            trailing = word[len(word.rstrip(PUNCT)):]
            if trailing and out:
                out[-1] = out[-1].rstrip(PUNCT) + trailing
            continue
        out.append(word)
    return (" ".join(out) + tail).strip()


def compression_ratio(text):
    """Tỉ lệ nén zlib — bắt lặp XEN KẼ mà unique_ratio bỏ sót.

    unique_ratio chỉ thấy lặp liên tiếp hoặc một từ áp đảo; chuỗi kiểu
    "tút tút loại tút thôi tút sút sút tút sút" có uniq 0.58 nên thoát. Whisper dùng cùng
    họ heuristic này với ngưỡng 2.4 cho đoạn ~30s tiếng Anh; chunk của ta ngắn hơn nhiều
    nên ngưỡng phải thấp hơn. Đo trên output thật của lần chạy trước:
        câu dùng được:  1.02 / 1.14 / 1.15
        câu rác:        1.45 / 2.10
    -> COMPRESSION_MAX = 1.30 nằm giữa, biên ~26%. Vẫn là heuristic, tinh chỉnh nếu cần:
    cột `zlib_ratio` trong CSV có sẵn để xem lại.
    """
    data = text.encode("utf-8")
    if len(data) < 20:            # chuỗi quá ngắn thì overhead của zlib áp đảo
        return 1.0
    return round(len(data) / len(zlib.compress(data, 9)), 3)


def unique_ratio(text):
    keys = [word_key(w) for w in text.split()]
    keys = [k for k in keys if k]
    if not keys:
        return 1.0
    return len(set(keys)) / len(keys)


def mean_logprob(model, generated):
    """Điểm tin cậy = logprob trung bình mỗi token sinh ra.

    compute_transition_scores là API có thể đổi giữa các bản transformers, nên bọc
    try/except: mất điểm tin cậy thì chỉ là không đánh dấu được, pipeline vẫn chạy.
    """
    try:
        scores = model.compute_transition_scores(
            generated.sequences, generated.scores, normalize_logits=True)
        values = scores[0]
        values = values[torch.isfinite(values)]
        return round(float(values.mean()), 3) if values.numel() else None
    except Exception:
        return None


# ============================================================
# Nạp model ASR
# ============================================================
if ASR_BACKEND == "phowhisper":
    from transformers import WhisperForConditionalGeneration, WhisperProcessor

    # KHÔNG dùng transformers.pipeline("automatic-speech-recognition"): nó có nhánh xử lý
    # long-form riêng, và ở transformers 5.0 nhánh đó bung `KeyError: 'num_frames'`.
    # Chunk của ta luôn <= MAX_SEGMENT_SEC (< 30s = cửa sổ của Whisper) nên chỉ cần một
    # forward pass chuẩn — vừa tránh lỗi, vừa nhanh hơn vì không qua DataLoader.
    print(f"Đang tải {PHOWHISPER}...")
    _dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    processor = WhisperProcessor.from_pretrained(PHOWHISPER)
    try:
        # transformers >= 5 dùng `dtype`; các bản cũ hơn dùng `torch_dtype`
        asr_model = WhisperForConditionalGeneration.from_pretrained(PHOWHISPER, dtype=_dtype)
    except TypeError:
        asr_model = WhisperForConditionalGeneration.from_pretrained(
            PHOWHISPER, torch_dtype=_dtype)
    asr_model = asr_model.to(DEVICE).eval()

    def transcribe(chunk):
        # processor tự pad/truncate về đúng cửa sổ 30s của Whisper
        features = processor(
            chunk, sampling_rate=SAMPLE_RATE, return_tensors="pt"
        ).input_features.to(device=DEVICE, dtype=_dtype)
        with torch.no_grad():
            generated = asr_model.generate(
                features,
                language=LANGUAGE,
                task="transcribe",
                num_beams=1,
                repetition_penalty=REPETITION_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM,
                return_dict_in_generate=True,
                output_scores=True,
            )
        text = processor.batch_decode(generated.sequences, skip_special_tokens=True)[0]
        return text.strip(), mean_logprob(asr_model, generated)

elif ASR_BACKEND == "whisper":
    try:
        import whisper
    except ImportError as exc:
        raise ImportError(
            "Chưa cài openai-whisper. Đặt INSTALL_OPENAI_WHISPER = True ở cell 1, chạy lại "
            "cell đó (kernel sẽ restart), rồi chạy lại từ cell 1b."
        ) from exc

    print(f"Đang tải whisper {WHISPER_SIZE}...")
    asr_model = whisper.load_model(WHISPER_SIZE, device=DEVICE)

    def transcribe(chunk):
        out = asr_model.transcribe(
            chunk,
            language=LANGUAGE,
            fp16=(DEVICE == "cuda"),
            condition_on_previous_text=False,          # không kéo hallucination sang chunk sau
            no_speech_threshold=0.6,
            logprob_threshold=AVG_LOGPROB_MIN,
            compression_ratio_threshold=2.4,           # loại output lặp bất thường
            temperature=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
        )
        parts = out.get("segments") or []
        logprob = round(float(np.mean([p["avg_logprob"] for p in parts])), 3) if parts else None
        return out["text"].strip(), logprob

else:
    raise ValueError(f"ASR_BACKEND phải là 'phowhisper' hoặc 'whisper', nhận: {ASR_BACKEND!r}")


# ============================================================
# Chạy ASR từng segment + chấm điểm
# ============================================================
dialogues = []
for idx, seg in enumerate(segments):
    track = SPEAKER_TRACKS.get(seg["speaker"])       # None khi không tách nguồn
    chunk = trim_silence(extract_chunk(seg["keep"], track))
    speech_sec = round(len(chunk) / sr, 2)

    flags, zlib_ratio = [], 1.0
    if seg.get("overlapped"):
        flags.append("chồng lấn")
    if speech_sec < MIN_SPEECH_SEC:
        # Không đưa vào ASR: chunk gần như không có tiếng thì nó chỉ bịa ra từ đơn.
        text, logprob = "", None
        flags.append("không có tiếng nói")
    else:
        text, logprob = transcribe(chunk)
        shortened = collapse_repeats(text)
        if shortened != text:
            flags.append("đã thu gọn lặp")
            text = shortened
        if not text:
            flags.append("không có tiếng nói")
        else:
            zlib_ratio = compression_ratio(text)
            repetitive = (len(text.split()) >= 4 and unique_ratio(text) < MIN_UNIQUE_RATIO)
            if repetitive or zlib_ratio > COMPRESSION_MAX:
                flags.append("lặp")
            if logprob is not None and logprob < AVG_LOGPROB_MIN:
                flags.append("tin cậy thấp")

    chunk_path = ""
    if len(chunk):
        chunk_path = os.path.join(OUTPUT_DIR, f"chunk_{idx:03d}_{seg['speaker']}.wav")
        sf.write(chunk_path, chunk, sr)

    dialogues.append({
        "start": seg["start"],
        "end": seg["end"],
        "duration": round(seg["end"] - seg["start"], 2),
        "speech_sec": speech_sec,
        "speaker": seg["speaker"],
        "text": text,
        "avg_logprob": logprob,
        "zlib_ratio": zlib_ratio,
        "peak_dbfs": dbfs(chunk),
        "n_parts": len(seg["keep"]),
        "separated": bool(SPEAKER_TRACKS),
        "flags": ", ".join(flags),
        "audio_path": chunk_path,
    })

    mark = f"  [{', '.join(flags)}]" if flags else ""
    print(f"  [{idx + 1}/{len(segments)}] {seg['speaker']} "
          f"{seg['start']:.2f}-{seg['end']:.2f}s ({speech_sec:.2f}s tiếng, "
          f"lp={logprob if logprob is not None else 'n/a'}): "
          f"{text or '(không có tiếng nói)'}{mark}")

clean = [d for d in dialogues if d["text"] and not d["flags"]]
scored = [d["avg_logprob"] for d in dialogues if d["avg_logprob"] is not None]
print(f"\n-> {len(dialogues)} segment: {len(clean)} sạch, "
      f"{len(dialogues) - len(clean)} có cảnh báo")
for name in ("chồng lấn", "không có tiếng nói", "tin cậy thấp", "lặp", "đã thu gọn lặp"):
    hit = sum(1 for d in dialogues if name in d["flags"])
    if hit:
        print(f"   {name}: {hit}")
if scored:
    print(f"   logprob trung bình: {np.mean(scored):.3f} "
          f"(ngưỡng cảnh báo {AVG_LOGPROB_MIN})")

for _name in ("asr_model", "processor"):   # tên khác nhau tuỳ backend
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Đang tải vinai/PhoWhisper-large...


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/1260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to proj_out.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['output_scores']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it 

  [1/11] SPEAKER_00 4.91-8.48s (3.57s tiếng, lp=-0.425): tút tút sút tóc.
  [2/11] SPEAKER_00 8.96-11.91s (2.95s tiếng, lp=-0.071): anh anh cho cho em xin cái đó đi hồ nước em đọc cho.  [đã thu gọn lặp]
  [3/11] SPEAKER_00 12.45-15.66s (3.21s tiếng, lp=-0.228): hồi nữa em đọc hồ nữa em báo cái em đ�c cho.
  [4/11] SPEAKER_00 15.83-17.11s (1.28s tiếng, lp=-0.023): ở trong này.
  [5/11] SPEAKER_00 17.14-21.65s (4.51s tiếng, lp=-0.037): cho em một công chuyện em làm đi trong này em buồn quá.
  [6/11] SPEAKER_00 21.51-27.48s (2.55s tiếng, lp=-0.246): lời đi.  [chồng lấn]
  [7/11] SPEAKER_00 27.18-32.23s (4.85s tiếng, lp=-0.297): anh túa a rồi chưa anh rồi anh tú a tuốt loại.
  [8/11] SPEAKER_01 22.93-26.35s (3.42s tiếng, lp=-0.142): đọc lại đi đọc.  [chồng lấn]
  [9/11] SPEAKER_00 33.31-35.80s (2.49s tiếng, lp=-0.262): anh tú a tút loại mày chưa nghe thấy này hay.
  [10/11] SPEAKER_00 37.60-42.28s (4.68s tiếng, lp=-0.412): ha ha.  [đã thu gọn lặp]
  [11/11] SPEAKER_00 42.32-46.96s (4.64s t

## 6. Hiển thị + lưu CSV

Segment có cảnh báo được làm mờ và gắn chip màu (`tin cậy thấp`, `lặp`,
`không có tiếng nói`, `đã thu gọn lặp`) chứ không bị xoá — bạn nghe lại rồi tự quyết.
Mỗi dòng hiện thêm số giây tiếng nói thực (sau khi cắt lặng), điểm `logprob`, và số mảnh
nếu segment bị chồng lấn chia nhỏ.

Xuất **hai** file: `dialogues_with_audio.csv` (đầy đủ, kèm cột `avg_logprob`, `peak_dbfs`,
`flags`) và `dialogues_with_audio_clean.csv` (chỉ câu không cảnh báo) để dùng cho bước sau
mà không phải lọc lại bằng tay.

In [9]:
import pandas as pd
from IPython.display import display, HTML, Audio

SPEAKER_PALETTE = [
    ("#e3f2fd", "#0d47a1"),   # xanh
    ("#f3e5f5", "#4a148c"),   # tím
    ("#e8f5e9", "#1b5e20"),   # lục
    ("#fff3e0", "#e65100"),   # cam
    ("#fce4ec", "#880e4f"),   # hồng
    ("#eceff1", "#263238"),   # xám
]
FLAG_STYLE = {
    "chồng lấn":          ("#e8eaf6", "#283593"),
    "không có tiếng nói": ("#eceff1", "#546e7a"),
    "tin cậy thấp":       ("#fff8e1", "#e65100"),
    "lặp":                ("#ffebee", "#b71c1c"),
    "đã thu gọn lặp":     ("#f1f8e9", "#33691e"),
}
speaker_color = {spk: SPEAKER_PALETTE[i % len(SPEAKER_PALETTE)]
                 for i, spk in enumerate(sorted({d["speaker"] for d in dialogues}))}


def badge(label, bg, fg, extra=""):
    return (f'<span style="background:{bg}; color:{fg}; font-size:.78em; font-weight:600;'
            f' padding:2px 7px; border-radius:10px; white-space:nowrap; {extra}">{label}</span>')


print(f"DANH SÁCH CÂU THOẠI ({len(dialogues)} segment)")
print("=" * 80)

for item in dialogues:
    bg, fg = speaker_color[item["speaker"]]
    flags = [f for f in item["flags"].split(", ") if f]
    chips = "".join(badge(f, *FLAG_STYLE.get(f, ("#f5f5f5", "#555"))) for f in flags)
    body = item["text"] or "<em style='color:#999'>(không có tiếng nói)</em>"
    faded = "opacity:.55;" if flags else ""
    lp = item["avg_logprob"]
    meta = (f"{item['speech_sec']:.2f}s tiếng"
            + (f" · lp {lp:+.2f}" if lp is not None else "")
            + (f" · {item['n_parts']} mảnh" if item["n_parts"] > 1 else ""))

    display(HTML(f"""
    <div style="border-bottom:1px solid #e0e0e0; padding:8px 0; display:flex;
                align-items:flex-start; gap:12px; font-family:sans-serif; {faded}">
      <span style="color:#666; font-size:.85em; width:120px; flex:none;">
        ⏱️ {item['start']:.2f}s – {item['end']:.2f}s<br>
        <span style="font-size:.85em; color:#999;">{meta}</span></span>
      <span style="background:{bg}; color:{fg}; font-weight:bold; padding:4px 8px;
                   border-radius:4px; min-width:110px; flex:none; text-align:center;">
        {item['speaker']}</span>
      <span style="flex:1; color:#222;">{body}
        <div style="margin-top:4px; display:flex; gap:5px; flex-wrap:wrap;">{chips}</div>
      </span>
    </div>"""))
    if item["audio_path"]:
        display(Audio(item["audio_path"]))

print("=" * 80)

df = pd.DataFrame(dialogues)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

# Bản chỉ giữ câu dùng được — tiện cho bước sau, không phải lọc lại bằng tay.
CLEAN_CSV = OUTPUT_CSV.replace(".csv", "_clean.csv")
df_clean = df[(df["text"].str.len() > 0) & (df["flags"] == "")]
df_clean.to_csv(CLEAN_CSV, index=False, encoding="utf-8-sig")

if os.path.exists(WAV_TEMP):
    os.remove(WAV_TEMP)

print(f"\nĐã lưu {len(df)} câu thoại  -> {OUTPUT_CSV}")
print(f"Trong đó {len(df_clean)} câu không cảnh báo -> {CLEAN_CSV}")
print(f"Các file chunk -> {OUTPUT_DIR}")
display(df[["start", "end", "speech_sec", "speaker", "avg_logprob", "zlib_ratio",
            "flags", "text"]])

DANH SÁCH CÂU THOẠI (11 segment)



Đã lưu 11 câu thoại  -> /kaggle/working/dialogues_with_audio.csv
Trong đó 7 câu không cảnh báo -> /kaggle/working/dialogues_with_audio_clean.csv
Các file chunk -> /kaggle/working/speaker_chunks


,start,end,speech_sec,speaker,avg_logprob,zlib_ratio,flags,text
0,4.910,8.480,3.57,SPEAKER_00,-0.425,0.952,,tút tút sút tóc.
1,8.960,11.910,2.95,SPEAKER_00,-0.071,1.000,đã thu gọn lặp,anh anh cho cho em xin cái đó đi hồ nước em đọ...
2,12.450,15.660,3.21,SPEAKER_00,-0.228,1.053,,hồi nữa em đọc hồ nữa em báo cái em đ�c cho.
3,15.830,17.110,1.28,SPEAKER_00,-0.023,1.000,,ở trong này.
4,17.140,21.650,4.51,SPEAKER_00,-0.037,0.930,,cho em một công chuyện em làm đi trong này em ...
5,21.510,27.479,2.55,SPEAKER_00,-0.246,1.000,chồng lấn,lời đi.
6,27.179,32.230,4.85,SPEAKER_00,-0.297,1.036,,anh túa a rồi chưa anh rồi anh tú a tuốt loại.
7,22.930,26.350,3.42,SPEAKER_01,-0.142,0.857,chồng lấn,đọc lại đi đọc.
8,33.310,35.800,2.49,SPEAKER_00,-0.262,0.915,,anh tú a tút loại mày chưa nghe thấy này hay.
9,37.600,42.280,4.68,SPEAKER_00,-0.412,1.000,đã thu gọn lặp,ha ha.


---
## Sự cố thường gặp

**`ImportError: cannot import name '_center' from 'numpy._core.umath'`**
(hoặc `numpy.dtype size changed`, `binary incompatibility`)
numpy bị đổi version dưới chân kernel đang chạy. Pip không sửa được trong cùng session:
**Factory reset** session rồi chạy lại từ cell 1 (cell 1 đã ghim numpy qua constraints).

**`AttributeError: module 'torchaudio' has no attribute 'AudioMetaData'`**
(hoặc traceback có `pytorch_lightning`)
`pyannote.audio` đang là bản 2.x của image, quá cũ. Chạy lại cell 1 — nó phát hiện version
thấp hơn 4.0 và nâng cấp, rồi restart kernel.

**`ResolutionImpossible` / `conflicting dependencies` khi cài pyannote.audio**
Đọc đoạn `The conflict is caused by:` mà cell 1 in ra. Cách xử lý phổ biến nhất là hạ
xuống nhánh 3.x: ở cell 1 đặt `PYANNOTE_SPEC = "pyannote.audio>=3.1,<4"`,
`PYANNOTE_MIN = "3.1"`, và ở cell 2 đổi `DIAR_MODEL = "pyannote/speaker-diarization-3.1"`.

**`ImportError: Numba needs NumPy 2.0 or less. Got NumPy 2.x`**
Bung ra khi gọi `librosa.load`. Pipeline đã bỏ librosa (cell 5 đọc WAV bằng `soundfile`),
nên nếu vẫn thấy lỗi này thì bạn đang chạy bản notebook cũ. Chỉ `ASR_BACKEND="whisper"` là
còn cần numba.

**`ModuleNotFoundError: No module named 'pyannote'`**
Cell 1 chưa chạy trong session này. Lưu ý **factory reset xoá sạch package đã cài** — sau mỗi
lần reset phải chạy lại cell 1 và tải lại torch 2.8.0 (~2-3GB). Chỉ reset khi numpy lệch ABI.

**`KeyError: 'num_frames'` khi transcribe**
Nhánh long-form của `transformers.pipeline` bị lỗi ở transformers 5.0. Cell 5 đã bỏ pipeline
và gọi trực tiếp `WhisperProcessor` + `model.generate`, nên nếu còn thấy thì bạn đang chạy
bản notebook cũ.

**`403 Forbidden ... Discussions are disabled for this repo` (thread `auto_conversion`)**
Vô hại. PhoWhisper chỉ có `pytorch_model.bin` (không có safetensors) nên transformers thử tìm
PR chuyển đổi trên Hub và bị chặn. Weights vẫn nạp bình thường, traceback đó ở thread phụ.

**`Pipeline.from_pretrained` trả về `None`**
Chưa bấm chấp nhận điều khoản tại trang model trên Hugging Face, hoặc token thiếu quyền đọc.

**`CUDA out of memory` khi nạp PhoWhisper-large**
Đảm bảo cell 4 đã chạy xong (nó tự `empty_cache()`), hoặc đổi sang
`PHOWHISPER = "vinai/PhoWhisper-medium"`.

**Không tìm thấy `HF_TOKEN`**
Add-ons → Secrets → tạo secret nhãn `HF_TOKEN` rồi bấm **Attach** cho notebook này.

---
### Ghi chú khi commit

Output của cell 6 nhúng toàn bộ audio dưới dạng base64 — file `.ipynb` có thể phình lên
vài MB và mang theo nội dung hội thoại thật. **Clear all outputs** trước khi commit:

```bash
jupyter nbconvert --clear-output --inplace t-ch-voice-v1.ipynb
```